In [ ]:
# 在 Google Colab 里运行时，先把本书的代码拉下来（本地 Jupyter 会跳过这一格）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/pytorch-inside-out"):
        !git clone -q --depth 1 https://github.com/wikieden/pytorch-inside-out /content/pytorch-inside-out
    %cd /content/pytorch-inside-out/code/notebooks


# 第 16 章 · 进程模型：一张卡一个进程，靠编号分工

本章回答：多卡训练时到底有几个 Python 进程？rank、world_size、local_rank 分别是什么？`torchrun` 帮我们做了什么？NCCL 和 Gloo 怎么选？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/16-process-model.md`（讲解、动画、练习都在讲义里）。对应原文：[00-overview.md](../00-overview.md) §5.1；AISPE ch03 DDP 示例（`ai-infra/materials/ai-systems-performance-engineering/docs/chapters/ch03.md` 第 263–440 行）；AISPE ch13「使用 PyTorch 分布式扩展」（`ch13.md` 第 1562–1619 行）；官方文档 torchrun（〔在线〕）；本章输出在本机用 Gloo 后端起 2 个进程跑过

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
import sys
sys.path.insert(0, "..")   # 子进程要能按名字导入函数，所以用本地模块

import os

from torch.utils.data import DistributedSampler
import torch
import torch.distributed as dist

import ch16_process_model as M          # 传给 run(...) 的函数从这里取
from dist_util import run
torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 rank 编号 | node × nproc + local | `global_rank(1, 4, 2)` | 6 |
| 第三步 两个进程报到 | torchrun 设的环境变量 | `run(whoami)` | rank 0 / 1，local 0 / 1 |
| 第四步 Sampler | 隔着分 | `sampler_indices(100, 8, r)` | rank 0 [0, 8, 16]，rank 1 [1, 9, 17] |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `global_rank`**（来自 `ch16_process_model.py`）：下面要用到的小工具。

In [ ]:
def global_rank(node_rank: int, nproc_per_node: int, local_rank: int) -> int:
    return node_rank * nproc_per_node + local_rank

**函数 `sampler_indices`**（来自 `ch16_process_model.py`）：DistributedSampler（不打乱）分给 rank 的前 k 个下标。

In [ ]:
def sampler_indices(n: int, world: int, rank: int, k: int = 3) -> list:
    """DistributedSampler（不打乱）分给 rank 的前 k 个下标。"""
    s = DistributedSampler(list(range(n)), num_replicas=world, rank=rank, shuffle=False)
    return list(s)[:k]

**函数 `whoami`**（来自 `ch16_process_model.py`）：下面要用到的小工具。

In [ ]:
def whoami(rank, world):
    return dict(rank=dist.get_rank(), world=dist.get_world_size(), local=os.environ["LOCAL_RANK"],
                master=f'{os.environ["MASTER_ADDR"]}:{os.environ["MASTER_PORT"] != ""}')

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 机器 1 上 local_rank 2（每台 4 卡）→ rank", global_rank(1, 4, 2))

In [ ]:
print("\n第三步 两个进程各自报到：", run(M.whoami))

In [ ]:
print("\n第四步 8 卡时 rank 0 / 1 拿到的前 3 个下标：", sampler_indices(100, 8, 0), sampler_indices(100, 8, 1))

## 3. 代码里才看得到的

1. **代码里用 `mp.spawn` 代替 torchrun**：效果一样，都是起 N 个进程、设好环境变量；测试里更方便。
2. **传给 `run` 的函数必须在模块顶层**：子进程要能按名字导入它。

## 4. 自己改一改

- `run(whoami, world=4)` 起 4 个进程。
- `sampler_indices(10, 4, 3)` 看样本数不能整除时怎么补。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 一卡一进程，同一份脚本，靠编号分工；
> world_size 总人数，rank 全局号，local_rank 定用哪张卡，rank 0 管日志；
> torchrun 起进程发编号、主节点会合；GPU 用 NCCL，Sampler 隔着分，每轮 set_epoch。